In [ ]:
# ============================================================
# CELL 1: KIỂM TRA MÔI TRƯỜNG
# Mục đích:
# - Kiểm tra phiên bản PyTorch
# - Kiểm tra torchvision
# - Kiểm tra Colab đang chạy CPU hay GPU
# ============================================================

import torch
import torchvision

print("PyTorch version    :", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("Device             :", "cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print("GPU                :", torch.cuda.get_device_name(0))

PyTorch version    : 2.11.0+cpu
Torchvision version: 0.26.0+cpu
Device             : cpu


In [ ]:
# ============================================================
# CELL 2: CÀI PYTORCH-OOD
# Mục đích:
# - Cài thư viện pytorch-ood để sử dụng implementation ViM
# ============================================================

!pip install -q pytorch-ood

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 635.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 473.2/473.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 12.4 MB/s eta 0:00:00


In [ ]:
# ============================================================
# CELL 3: IMPORT THƯ VIỆN
# Mục đích:
# - Import các thư viện cần cho ViM
# - Thiết lập device
# ============================================================

import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader
from torchvision import datasets

from pytorch_ood.detector import ViM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cpu


In [ ]:
# ============================================================
# CELL 4: TẢI DATASET
# Mục đích:
# - CIFAR-10 train/test: dữ liệu ID
# - CIFAR-100 test: OOD dataset thứ nhất
# - MNIST test: OOD dataset bổ sung
# ============================================================

# CIFAR-10 preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

# MNIST is grayscale and 28x28. The CIFAR-10 model expects 3-channel 32x32
# images, so resize MNIST to 32x32, convert it to 3 channels, and then apply
# the same normalization used for CIFAR-10.
mnist_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

# CIFAR-10 TRAIN (ID data used to fit ViM)
cifar10_train = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

# CIFAR-10 TEST (ID)
cifar10_test = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# CIFAR-100 TEST (OOD dataset 1)
cifar100_test = datasets.CIFAR100(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# MNIST TEST (additional OOD dataset)
mnist_test = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=mnist_transform
)

print("CIFAR-10 train:", len(cifar10_train))
print("CIFAR-10 test :", len(cifar10_test))
print("CIFAR-100 test:", len(cifar100_test))
print("MNIST test     :", len(mnist_test))


100%|██████████| 170M/170M [01:14<00:00, 2.29MB/s]
100%|██████████| 169M/169M [00:36<00:00, 4.60MB/s]
100%|██████████| 9.91M/9.91M [00:01<00:00, 5.30MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 153kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.45MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.37MB/s]

CIFAR-10 train: 50000
CIFAR-10 test : 10000
CIFAR-100 test: 10000
MNIST test     : 10000


In [ ]:
# ============================================================
# CELL 5: TẢI WIDERESNET-40-2
# Mục đích:
# - Tải checkpoint WideResNet đã pretrained trên CIFAR-10
# - Dùng cùng model với MSP và KNN
# ============================================================

import urllib.request

checkpoint_url = (
    "https://github.com/wetliu/energy_ood/raw/master/"
    "CIFAR/snapshots/pretrained/cifar10_wrn_pretrained_epoch_99.pt"
)

checkpoint_path = "./cifar10_wrn_pretrained_epoch_99.pt"

urllib.request.urlretrieve(
    checkpoint_url,
    checkpoint_path
)

print("Checkpoint downloaded:", checkpoint_path)

Checkpoint downloaded: ./cifar10_wrn_pretrained_epoch_99.pt


In [ ]:
# ============================================================
# CELL 6: LOAD WIDERESNET
# Mục đích:
# - Tạo kiến trúc WideResNet-40-2
# - Load trọng số pretrained
# - Đưa model về evaluation mode
# ============================================================

from pytorch_ood.model import WideResNet

model = WideResNet(
    num_classes=10,
    depth=40,
    widen_factor=2
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device
)

model.load_state_dict(checkpoint)
model = model.to(device)
model.eval()

print(model)

WideResNet(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (block1): NetworkBlock(
    (layer): Sequential(
      (0): BasicBlock(
        (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (convShortcut): Conv2d(16, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
      )
      (1): BasicBlock(
        (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(32, 

In [ ]:
# ============================================================
# CELL 7: KIỂM TRA FEATURE VÀ LOGITS
# Mục đích:
# - Lấy một batch CIFAR-10
# - Kiểm tra feature có 128 chiều không
# - Kiểm tra logits có 10 chiều không
# ============================================================

test_loader = DataLoader(
    cifar10_test,
    batch_size=8,
    shuffle=False
)

images, labels = next(iter(test_loader))
images = images.to(device)

with torch.no_grad():
    features = model.features(images)
    logits = model(images)

print("Input shape   :", images.shape)
print("Feature shape :", features.shape)
print("Logits shape  :", logits.shape)

Input shape   : torch.Size([8, 3, 32, 32])
Feature shape : torch.Size([8, 128])
Logits shape  : torch.Size([8, 10])


In [ ]:
# ============================================================
# CELL 8: KHỞI TẠO ViM
# Mục đích:
# - Tạo detector ViM
# - d=64: sử dụng principal subspace 64 chiều
# - Dùng chính feature và FC của WideResNet
# ============================================================

vim = ViM(
    model.features,
    d=64,
    w=model.fc.weight,
    b=model.fc.bias
)

vim = vim.to(device)

print(vim)

ViM(d=64)


In [ ]:
# ============================================================
# CELL 9: TẠO TRAIN DATALOADER CHO ViM
# Mục đích:
# - Dùng 50,000 ảnh CIFAR-10 train
# - ViM sẽ dùng feature của các ảnh này để
#   xây dựng principal subspace và tính alpha
# ============================================================

cifar10_train_loader = DataLoader(
    cifar10_train,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

print("CIFAR-10 train images :", len(cifar10_train))
print("Number of batches     :", len(cifar10_train_loader))

CIFAR-10 train images : 50000
Number of batches     : 391


In [ ]:
# ============================================================
# CELL 10: FIT ViM
# Mục đích:
# - Chạy 50,000 ảnh CIFAR-10 train qua model
# - Lấy feature 128 chiều
# - ViM dùng các feature này để xác định principal subspace
# - Đồng thời tính hệ số alpha cần cho ViM
# ============================================================

print("Starting ViM fitting...")

vim.fit(cifar10_train_loader)

print("ViM fitting completed.")

Starting ViM fitting...
ViM fitting completed.


In [ ]:
# ============================================================
# CELL 11: TEST ViM TRÊN 1 ẢNH ID VÀ CÁC ẢNH OOD
# Mục đích:
# - Lấy 1 ảnh CIFAR-10 test (ID)
# - Lấy 1 ảnh CIFAR-100 và 1 ảnh MNIST (OOD)
# - Kiểm tra ViM score cho từng ảnh
# ============================================================

id_image, id_label = cifar10_test[0]
cifar100_image, cifar100_label = cifar100_test[0]
mnist_image, mnist_label = mnist_test[0]

id_image = id_image.unsqueeze(0).to(device)
cifar100_image = cifar100_image.unsqueeze(0).to(device)
mnist_image = mnist_image.unsqueeze(0).to(device)

with torch.no_grad():
    id_score = vim(id_image)
    cifar100_score = vim(cifar100_image)
    mnist_score = vim(mnist_image)

print("ID (CIFAR-10) ViM score :", id_score.item())
print("OOD (CIFAR-100) score   :", cifar100_score.item())
print("OOD (MNIST) ViM score   :", mnist_score.item())

print("\nCIFAR-10 label :", id_label)
print("CIFAR-100 label:", cifar100_label)
print("MNIST label    :", mnist_label)


ID (CIFAR-10) ViM score : -2.8683881759643555
OOD (CIFAR-100) score   : 2.5934133529663086
OOD (MNIST) ViM score   : 7.930790901184082

CIFAR-10 label : 3
CIFAR-100 label: 49
MNIST label    : 7


In [ ]:
# ============================================================
# CELL 12: TÍNH ViM SCORE CHO TOÀN BỘ ID
# Mục đích:
# - Chạy 10,000 ảnh CIFAR-10 test qua ViM
# - Lưu ViM score của từng ảnh
# - Dùng các score này để đánh giá OOD detection
# ============================================================

id_loader = DataLoader(
    cifar10_test,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

id_scores = []

print("Calculating ViM scores for CIFAR-10 test...")

with torch.no_grad():
    for images, labels in id_loader:
        images = images.to(device)

        scores = vim(images)

        id_scores.append(scores.cpu())

id_scores = torch.cat(id_scores)

print("Done.")
print("Number of ID scores:", len(id_scores))
print("ID score shape      :", id_scores.shape)

print("\nID ViM statistics:")
print("Mean   :", id_scores.mean().item())
print("Median :", id_scores.median().item())
print("Min    :", id_scores.min().item())
print("Max    :", id_scores.max().item())

Calculating ViM scores for CIFAR-10 test...
Done.
Number of ID scores: 10000
ID score shape      : torch.Size([10000])

ID ViM statistics:
Mean   : 0.45226919651031494
Median : 0.0061206817626953125
Min    : -10.496024131774902
Max    : 21.38593292236328


In [ ]:
# ============================================================
# CELL 13: TÍNH ViM SCORE CHO TOÀN BỘ OOD
# Mục đích:
# - Chạy 10,000 ảnh CIFAR-100 test qua ViM
# - Lưu ViM score của từng ảnh
# ============================================================

ood_loader = DataLoader(
    cifar100_test,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

ood_scores = []

print("Calculating ViM scores for CIFAR-100 test...")

with torch.no_grad():
    for images, labels in ood_loader:
        images = images.to(device)

        scores = vim(images)

        ood_scores.append(scores.cpu())

ood_scores = torch.cat(ood_scores)

print("Done.")
print("Number of OOD scores:", len(ood_scores))
print("OOD score shape      :", ood_scores.shape)

print("\nOOD ViM statistics:")
print("Mean   :", ood_scores.mean().item())
print("Median :", ood_scores.median().item())
print("Min    :", ood_scores.min().item())
print("Max    :", ood_scores.max().item())

Calculating ViM scores for CIFAR-100 test...
Done.
Number of OOD scores: 10000
OOD score shape      : torch.Size([10000])

OOD ViM statistics:
Mean   : 5.7124505043029785
Median : 5.454236030578613
Min    : -7.667074203491211
Max    : 37.75872802734375


In [ ]:
# ============================================================
# CELL 14: TÍNH ViM SCORE CHO TOÀN BỘ MNIST (OOD)
# Mục đích:
# - Chạy toàn bộ 10,000 ảnh MNIST test qua ViM
# - Lưu score để đánh giá MNIST như một OOD dataset riêng
# ============================================================

mnist_loader = DataLoader(
    mnist_test,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

mnist_scores = []

print("Calculating ViM scores for MNIST test...")

with torch.no_grad():
    for images, labels_batch in mnist_loader:
        images = images.to(device)
        batch_scores = vim(images)
        mnist_scores.append(batch_scores.cpu())

mnist_scores = torch.cat(mnist_scores)

print("Done.")
print("Number of MNIST OOD scores:", len(mnist_scores))
print("MNIST score shape          :", mnist_scores.shape)

print("\nMNIST ViM statistics:")
print("Mean   :", mnist_scores.mean().item())
print("Median :", mnist_scores.median().item())
print("Min    :", mnist_scores.min().item())
print("Max    :", mnist_scores.max().item())


Calculating ViM scores for MNIST test...
Done.
Number of MNIST OOD scores: 10000
MNIST score shape          : torch.Size([10000])

MNIST ViM statistics:
Mean   : 7.505657196044922
Median : 7.524822235107422
Min    : -0.7094001770019531
Max    : 17.134004592895508


In [ ]:
# ============================================================
# CELL 14.5: KIỂM TRA KẾT QUẢ ViM OOD
# Mục đích:
# - Kiểm tra ood_scores đã được tạo
# - Kiểm tra số lượng score
# - Hiển thị một vài score đầu tiên
# - Hiển thị thống kê của toàn bộ OOD scores
# ============================================================

print("========== ViM OOD SCORES ==========")

print("Number of OOD scores:", len(ood_scores))
print("Shape               :", ood_scores.shape)

print("\nFirst 10 OOD scores:")
print(ood_scores[:10])

print("\nOOD ViM statistics:")
print("Mean   :", ood_scores.mean().item())
print("Median :", ood_scores.median().item())
print("Min    :", ood_scores.min().item())
print("Max    :", ood_scores.max().item())

========== ViM OOD SCORES ==========
Number of OOD scores: 10000
Shape               : torch.Size([10000])

First 10 OOD scores:
tensor([ 2.5934,  5.8383, 10.6776,  6.0521, 12.9633,  3.1263,  6.4386,  5.8314,
         6.3075, 13.3894])

OOD ViM statistics:
Mean   : 5.7124505043029785
Median : 5.454236030578613
Min    : -7.667074203491211
Max    : 37.75872802734375


In [ ]:
# ============================================================
# CELL 15: ĐÁNH GIÁ ViM OOD DETECTION
# Mục đích:
# - Đánh giá CIFAR-100 và MNIST riêng biệt
# - Đánh giá thêm kết quả khi gộp cả hai OOD datasets
# - ID có label = 0; OOD có label = -1
# ============================================================

from pytorch_ood.utils.metrics import OODMetrics

def evaluate_ood(id_scores, ood_scores, dataset_name):
    """Evaluate one OOD dataset against the same CIFAR-10 ID scores."""
    scores = torch.cat([id_scores, ood_scores])
    labels = torch.cat([
        torch.zeros(len(id_scores)),
        -torch.ones(len(ood_scores))
    ])

    metrics = OODMetrics()
    metrics.update(scores, labels)
    results = metrics.compute()

    print(f"\n===== ViM OOD RESULTS: {dataset_name} =====")
    print("Scores shape:", scores.shape)
    print("Labels shape:", labels.shape)
    for key, value in results.items():
        print(f"{key}: {value:.4f}")
    return results

# Evaluate each OOD dataset separately so their results can be compared fairly.
cifar100_results = evaluate_ood(id_scores, ood_scores, "CIFAR-100")
mnist_results = evaluate_ood(id_scores, mnist_scores, "MNIST")

# Also evaluate the union of the two OOD datasets.
combined_ood_scores = torch.cat([ood_scores, mnist_scores])
combined_results = evaluate_ood(
    id_scores,
    combined_ood_scores,
    "CIFAR-100 + MNIST"
)



===== ViM OOD RESULTS: CIFAR-100 =====
Scores shape: torch.Size([20000])
Labels shape: torch.Size([20000])
AUROC: 0.8240
AUTC: 0.4455
AUPR-IN: 0.8238
AUPR-OUT: 0.8075
FPR95TPR: 0.6170

===== ViM OOD RESULTS: MNIST =====
Scores shape: torch.Size([20000])
Labels shape: torch.Size([20000])
AUROC: 0.9329
AUTC: 0.3894
AUPR-IN: 0.9509
AUPR-OUT: 0.8879
FPR95TPR: 0.1941

===== ViM OOD RESULTS: CIFAR-100 + MNIST =====
Scores shape: torch.Size([30000])
Labels shape: torch.Size([30000])
AUROC: 0.8785
AUTC: 0.4362
AUPR-IN: 0.8032
AUPR-OUT: 0.9182
FPR95TPR: 0.4573
